# Lab 3 — Inspect & Talk to Models
**Day 1 Morning | ~45 minutes | Colab CPU**

---

## What You Will Learn
By the end of this lab you understand *why* these things matter, not just that they exist:

1. A transformer model has a fixed shape — layers, heads, context window — and those numbers have real deployment consequences
2. Generation is not magic: it is token-by-token probability sampling, and you control it with knobs
3. Instruct models have a contract: they expect a specific token format, not raw strings
4. A chat application is just a loop that manually manages a growing list of messages
5. Context windows fill up — and that is one of the most common silent production bugs

> **The key idea:** A model is not a text box. It has a contract — a specific token format, a finite working memory, and sampling knobs that are product decisions, not magic.

---

## The Flow of This Lab

```
Part A                  Part B                    Part C
──────────────          ──────────────────────    ─────────────────────────
Open the hood      →    Control generation    →   Build a real chat session
(architecture)          (greedy, temperature,     (history, context growth,
                         chat templates)           sliding window trim)
```

Each part has a **Concept** section (read it), a **Code** section (run it), and a **Checkpoint** (confirm you understand it before moving on).


In [ ]:
!uv pip install -q transformers torch accelerate
print('Install complete')


In [ ]:
# Configuration — local model only, no API key needed
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
print(f'Config loaded — using {MODEL_ID}')


---

### 🧠 Before We Start — What Is a Transformer Model, Really?

A language model like Qwen2.5-0.5B-Instruct is a stack of mathematical layers stored as a file on disk. When you call `from_pretrained(...)`, you are:

1. **Downloading a config file** — a JSON that describes the model's shape (layers, heads, dimensions)
2. **Downloading weights** — the actual numbers that were learned during training (~1 GB for this model)
3. **Loading them into RAM** — so Python can run computations on them

The model has no "memory" between calls. Every time you send it text, it starts fresh. The only memory it has is what you explicitly put in the prompt.

> **Key numbers you will read:** `hidden_size` (how wide each layer is), `num_hidden_layers` (how deep the stack is), `max_position_embeddings` (the maximum number of tokens it can process at once — its context window).


---

## Why Do We Inspect a Model?

You inspect a model for the same reason an engineer reads a datasheet before selecting a component: **to make decisions before they become expensive mistakes.** A model is not a text box — it is a system component with a fixed shape, a memory footprint, and hard limits. Before you deploy, you need to know those numbers.

**Five deployment decisions that require knowing your model's architecture:**

| Decision | What you need to know |
|---|---|
| **Hardware selection** (which GPU, how much VRAM) | Hidden size × layers × dtype → memory estimate |
| **Max concurrent users** (serving capacity) | KV heads (GQA ratio) → KV cache size per user → throughput |
| **Prompt and RAG design** | Context window → tokens available after system prompt + retrieved docs |
| **Quantization planning** | BF16 baseline → INT4/INT8 estimate → decide whether to quantize at all |
| **Model selection** | Compare hidden size, context window, GQA across candidates objectively |

---

### Can You Always Inspect? — Open Source vs. Closed Models

Not all models are inspectable. The answer depends on whether you have the weights.

| What you want to inspect | Open Source (Qwen, Llama, Gemma, Mistral) | Closed / API-only (GPT-4, Claude, Gemini) |
|---|---|---|
| Config: layers, hidden size, GQA ratio | ✅ `model.config` | ❌ Proprietary — not disclosed |
| Weights: quantize, fine-tune, self-host | ✅ You have the files | ❌ Provider hosts them |
| Context window | ✅ `config.max_position_embeddings` | ✅ Documented in API docs |
| KV cache behavior, GQA ratio | ✅ `config.num_key_value_heads` | ❌ Not disclosed |
| Memory footprint | ✅ Calculate from weights | N/A — provider pays for that RAM |
| Token pricing / cost per call | N/A — you pay for your own compute | ✅ Per-token pricing in docs |

> **The practical implication:** With a closed API (GPT-4, Claude), you design around the context window and estimate costs — but you *cannot* do capacity planning for self-hosting, because you don't have the weights. With an open source model, you can inspect everything and make informed decisions about hardware, quantization, and serving architecture.
>
> **Rule of thumb:** If you are self-hosting, you must be able to inspect. If you are using an API, you work with what the provider documents.

---

---

## Part A — Architecture Inspection (15 min)

We load `Qwen/Qwen2.5-0.5B-Instruct` — a modern, tiny instruct model (~1 GB on CPU). We are reading both the config and the actual module layout.

> ⏳ First download takes 2–3 minutes. Subsequent runs are instant because Colab caches weights.

**What to look for when the output appears:**
- `Hidden size` → the width of each layer (d_model). Bigger = more expressive, more memory.
- `Layers` → the depth of the stack. Each layer is one attention + MLP block.
- `Context window` → the hard limit on tokens per request. Beyond this, the model cannot see earlier text.
- `KV heads vs Attention heads` → if they differ, the model uses Grouped Query Attention (GQA), which reduces memory during inference.

**Before you run — make a prediction:**
*A "0.5B" model has approximately 500 million parameters. Given that, do you expect more or fewer than 10 transformer layers? Write your guess in the cell below before running.*


#### ✏️ Your Prediction

> My guess for number of layers: ___  
> My guess for context window size (in tokens): ___

*(Fill this in mentally or in a comment — you will check it against the output.)*


In [ ]:
# Cell A1 — Load tokenizer and model config
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id  = MODEL_ID
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16)
config    = model.config

attention_heads = config.num_attention_heads
kv_heads = getattr(config, 'num_key_value_heads', None)

print('KEY ARCHITECTURE NUMBERS')
print(f'  Model           : {model_id}')
print(f'  Hidden size     : {config.hidden_size}  (d_model)')
print(f'  Layers          : {config.num_hidden_layers}')
print(f'  Attention heads : {attention_heads}')
print(f'  KV heads (GQA)  : {kv_heads if kv_heads is not None else "N/A"}')
print(f'  Vocab size      : {config.vocab_size:,}')
print(f'  Context window  : {config.max_position_embeddings:,} tokens')

if kv_heads and kv_heads < attention_heads:
    ratio = attention_heads / kv_heads
    print()
    print('GQA CALLOUT')
    print(f'  This model uses fewer KV heads than attention heads ({attention_heads}:{kv_heads}, ratio {ratio:.1f}x).')
    print('  That is Grouped Query Attention: query heads share key/value heads.')
    print('  Deployment impact: smaller KV cache, lower memory pressure during long-context inference.')

params = sum(p.numel() for p in model.parameters())
print(f'\n  Parameters      : {params:,}  ({params/1e9:.2f}B)')
print(f'  Memory BF16     : ~{params*2/1e9:.2f} GB')
print(f'  Memory INT4     : ~{params*0.5/1e9:.2f} GB  ← what you will use in Lab 4')


In [ ]:
# Cell A2 — Layer structure: what is actually inside?
print('LAYER STRUCTURE (first 18 named modules)')
print(f'{"Name":<45} {"Type"}')
print('-' * 65)
for name, module in list(model.named_modules())[:18]:
    print(f'  {name:<43} {type(module).__name__}')
print('  ...')
print()
print('The repeating pattern (layers.0, layers.1, ...) is the transformer stack.')
print('Attention + MLP + LayerNorm — repeated num_hidden_layers times.')

<details>
<summary><b>🔍 Reading the Output — How to Interpret Each Architecture Number</b></summary>

**Answer to the prediction:** 24 layers, 32,768 token context window. Most people guess ~12 layers for a 0.5B model. This model is *deep and narrow* — more layers than expected, with a smaller hidden size than wider models. More depth = more reasoning steps per token; smaller width = lower compute and memory per layer. That trade-off is a deliberate design choice.

---

**Interpreting each number and the deployment decision it drives:**

| Number | This Model | Typical Range | What It Means for Deployment |
|---|---|---|---|
| **Hidden size** | 896 | 512 (tiny) → 7168 (13B+) | Each layer's width. Larger = more expressive but more memory and compute. 896 is narrow/efficient. |
| **Layers** | 24 | 12 (small) → 80+ (large) | Stack depth. More layers = more reasoning steps per token. Surprisingly deep for 0.5B — expect latency proportional to depth. |
| **Attention heads** | 14 | 8 → 64 | Parallel attention patterns per layer. Head dimension = 896 ÷ 14 = 64. Each head learns different token relationships. |
| **KV heads (GQA)** | 2 | 1 → equal to attn heads | ⚡ **Most deployment-critical number.** 14 query heads share 2 KV heads = 7:1 ratio. KV cache is 7× smaller than full attention. Directly controls how many concurrent users you can serve. |
| **Context window** | 32,768 | 2K → 1M+ | Hard limit (input + output combined). ~25,000 words. Beyond this: **silent truncation** — no error, the model just stops seeing older tokens. |
| **Vocab size** | 151,936 | 32K → 256K | Size of the token dictionary. Large here because Qwen is multilingual. Larger vocab → better tokenization efficiency for non-English text. |
| **Memory BF16** | ~1 GB | — | Weights only. Actual runtime RAM = weights + activations (~0.5–1× more) + KV cache (grows with context length × batch size). |
| **Memory INT4** | ~0.25 GB | — | After 4-bit quantization (Lab 4). 4× compression — same model, 75% less VRAM. This is the first lever to pull when a model doesn't fit your hardware. |

---

**The GQA story — why KV heads matter more than attention heads at serving time:**

At each generation step the model reads (and stores) the key and value tensors for every token it has seen. This *KV cache* grows linearly with context length and batch size. It is the primary memory bottleneck in LLM serving — not the weights.

```
Full attention (MHA):   14 Q heads × 14 KV pairs per layer → large KV cache
GQA (this model):       14 Q heads ×  2 KV pairs per layer → 7× smaller KV cache
```

Smaller KV cache → more concurrent requests fit in the same GPU RAM → lower serving cost per user. This is why every modern production model (Llama 3, Mistral, Qwen 2.5, Gemma 2) uses GQA instead of full multi-head attention.

> **Decision rule:** When comparing models for self-hosted deployment, check `num_key_value_heads`. A lower ratio (e.g., 2 KV heads vs 14 attention heads) is a signal that the model was optimized for serving efficiency, not just benchmark accuracy.

</details>

#### ✅ Part A Checkpoint
- [ ] You have inspected the model config and printed the layers.\n
- [ ] You can see the model has 24 layers and a 32,768 token context window.\n

#### ✅ Part A Checkpoint

Confirm you can answer these before moving on:
- [ ] What is the context window of this model in tokens?
- [ ] What does `model.layers.0.self_attn.q_proj` represent? (Hint: q = query)
- [ ] What does the repeating `layers.0`, `layers.1`, `layers.2`... pattern mean?

> **Answer key:** Context window: 32,768 tokens. The repeating pattern is the transformer stack — each decoder layer contains one self-attention block and one MLP block, stacked `num_hidden_layers` times. The model does not have different "types" of layers — it has one design, repeated.


---

## Part B — Generation Controls (15 min)

Before you wire generation settings into a product, you need to understand what they actually do. This part answers:
- Why does the same prompt sometimes give different answers?
- What does "temperature" actually change?
- Why can't you just send a plain string to an instruct model?

### How Generation Works (Read This First)

When the model generates, it does not write a sentence — it picks **one token at a time**, over and over, until it hits a stop condition. At each step, it produces a probability distribution over the entire vocabulary (~151,000 tokens), then picks the next token from that distribution.

**Two strategies:**
- **Greedy:** Always pick the single highest-probability token. Deterministic — same input always gives same output.
- **Sampling:** Draw randomly from the distribution. `temperature` controls how "spread out" the distribution is before sampling.

```
temperature = 0.1  →  distribution is very peaked  →  safe, repetitive, predictable
temperature = 1.0  →  distribution is natural       →  balanced
temperature = 1.5  →  distribution is very flat     →  creative, sometimes incoherent
```


### Step B1 — The `generate()` Helper

The function below handles three things that happen on every generation call:
1. **Format the prompt** using the chat template (more on this in B3)
2. **Tokenize** — convert the formatted string to a list of integer IDs
3. **Decode** — convert the output token IDs back to a human-readable string

It also measures `input_len` so that it only returns the *new* tokens (the model's response), not the original prompt tokens.

*Run this cell — it defines the helper and immediately demonstrates greedy decoding. Run it twice to confirm the output is identical both times.*


### Tiny Tokenization Demo

Before generation, look at what "text" becomes to the model: a list of integer token IDs. The tokenizer is the adapter between human-readable strings and model-readable numbers.

*Run this cell and notice the round trip: text → token IDs → text again.*


In [ ]:
sample = tokenizer("Hello, I am a language model.")
print(sample["input_ids"])
print(tokenizer.decode(sample["input_ids"]))


In [ ]:
# Cell B1 — Helper: generate from local model
def generate(prompt_text, max_new_tokens=80, **kwargs):
    # If temperature is provided, sampling must be enabled or Transformers will warn/ignore it.
    if 'temperature' in kwargs and 'do_sample' not in kwargs:
        kwargs['do_sample'] = True

    msgs = [{'role': 'user', 'content': prompt_text}]
    formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs    = tokenizer(formatted, return_tensors='pt')
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id, **kwargs
        )
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)

q = 'What are the top 3 challenges in deploying LLMs to production? Be brief.'
print('GREEDY (deterministic) — run twice, expect identical output:')
print(f'Run 1: {generate(q, max_new_tokens=80, do_sample=False)[:120]}')
print(f'Run 2: {generate(q, max_new_tokens=80, do_sample=False)[:120]}')


### Step B2 — Temperature & Top-p

**Temperature** scales the logits (raw scores) before the softmax that produces probabilities. Lower temperature makes the model "more confident" — it concentrates probability on the top tokens.

**Top-p (nucleus sampling)** is a filter applied *after* temperature: it keeps only the smallest set of tokens whose cumulative probability adds up to `p`. At `top_p=0.9`, the model considers only the tokens that together account for 90% of the probability mass — ignoring the long tail of unlikely tokens.

*Watch the output carefully:*
- At `temp=0.1`: the model is cautious and structured
- At `temp=1.5`: look for odd word choices or broken formatting — that is what "overconfident randomness" looks like in practice

**Question to hold in mind:** Which temperature would you use for a customer support chatbot? Which for a creative writing tool?


In [ ]:
# Cell B2 — Temperature and top_p: production generation knobs
test = 'List 3 steps to debug slow LLM inference in production:'
print('TEMPERATURE COMPARISON (same deployment prompt, four temperatures):\n')
for temp in [0.1, 0.5, 1.0, 1.5]:
    out = generate(test, max_new_tokens=55, temperature=temp)
    print(f'  temp={temp}: {out[:180]}\n')

print('=' * 70)
print('TOP-P COMPARISON (same temperature, narrower nucleus sampling):\n')
wide = generate(test, max_new_tokens=55, temperature=0.9, top_p=1.0)
narrow = generate(test, max_new_tokens=55, temperature=0.9, top_p=0.9)
print(f'  top_p=1.0: {wide[:180]}\n')
print(f'  top_p=0.9: {narrow[:180]}\n')
print('Deployment note: lower temperature/top_p is usually safer for support, tools, and structured output.')


#### Reflection — Choose the Knob for the Job

> **Now that you've seen the outputs:** `temp=0.1` is usually better for support because it is consistent and safer. `temp=1.0–1.5` fits brainstorming or creative writing, where variety is useful. The "Deployment note" at the bottom of the output confirms this.


### Step B3 — Why Chat Templates Exist

An instruct model is **not** a text-completion model. It was fine-tuned on conversations that always had a specific format using special tokens like `<|im_start|>` and `<|im_end|>`. If you send it a raw string, it does not know what role is speaking — it may ignore your instruction entirely or produce garbage.

The `apply_chat_template()` function takes a list of `{"role": ..., "content": ...}` dicts and wraps them in the format the model was trained on. This is the model's **contract** — it must be honored.

*The output will show you three versions of the same question:*
1. Raw string — what a naive caller might send
2. User-only template — the minimum correct format
3. System + user template — what production systems use

**Look at the special tokens in the output.** The `<|im_start|>` and `<|im_end|>` markers are literally part of the model's vocabulary — they are not decorative punctuation.


In [ ]:
# Cell B3 — Why chat templates exist
raw_prompt = 'What is quantization?'

formatted  = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': raw_prompt}],
    tokenize=False, add_generation_prompt=True
)

msgs_with_system = [
    {'role': 'system', 'content': 'You are an LLM deployment expert.'},
    {'role': 'user', 'content': raw_prompt},
]
formatted_with_system = tokenizer.apply_chat_template(
    msgs_with_system, tokenize=False, add_generation_prompt=True
)

print('Raw string sent to model:')
print(f'  {raw_prompt!r}')
print()
print('Formatted user-only chat template:')
print(repr(formatted))
print()
print('Formatted system + user chat template:')
print(repr(formatted_with_system))
print()
print('The im_start / im_end markers are the trained signal for instruction-following.')
print('System and user roles become different segments in the formatted prompt.')


In [ ]:
# Let's actually generate from both to see the difference in output

raw_inputs = tokenizer(raw_prompt, return_tensors='pt')
out_raw = model.generate(**raw_inputs, max_new_tokens=50)
print('RAW STRING OUTPUT:')
print(tokenizer.decode(out_raw[0][raw_inputs['input_ids'].shape[1]:], skip_special_tokens=True))
print('\n' + '='*60 + '\n')

fmt_inputs = tokenizer(formatted, return_tensors='pt')
out_fmt = model.generate(**fmt_inputs, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
print('TEMPLATED OUTPUT:')
print(tokenizer.decode(out_fmt[0][fmt_inputs['input_ids'].shape[1]:], skip_special_tokens=True))


#### ✅ Part B Checkpoint

- [ ] Greedy decoding is deterministic: running the same prompt twice gives the same output
- [ ] Temperature 0.1 is conservative; temperature 1.5 can produce broken or off-topic output
- [ ] You can see the `<|im_start|>` / `<|im_end|>` special tokens in the chat template output
- [ ] You understand why sending a raw string to an instruct model is wrong


---

## Part C — Multi-Turn Conversation (15 min)

### The Core Mental Model — Read This Before Running Anything

A chat application does **not** have memory. The model has no state between API calls. What looks like "memory" is an illusion you create by sending the entire conversation history on every single request.

Here is what actually happens on each turn:

```
Turn 1:  [system] + [user: "What is quantization?"]
         → sent to model → [assistant: "..."]

Turn 2:  [system] + [user: "What is quantization?"] + [assistant: "..."] + [user: "Compare to pruning?"]
         → sent to model → [assistant: "..."]

Turn 3:  [system] + ALL PREVIOUS TURNS + [user: "Which first for 7B?"]
         → sent to model → [assistant: "..."]
```

Every turn, the full transcript grows. This has two consequences:
1. **Cost** — you pay for more input tokens each turn (for hosted APIs)
2. **Context pressure** — eventually the history fills the context window and old turns are cut off

The `ChatSession` class below is the minimal implementation of this pattern.


### Step C1 — The `ChatSession` Class

Read through the class before running it. Each method does one thing:

- `__init__` — starts history with the system prompt
- `chat(message)` — appends user message → formats → generates → appends assistant reply → returns reply
- `token_estimate()` — counts how many tokens the full current history uses
- `show()` — prints a readable preview of all messages in history
- `show_full_prompt()` — prints the **exact raw string** sent to the model (very useful for debugging)
- `trim(max_turns=N)` — keeps system prompt + only the last N user/assistant pairs (the sliding window strategy)

> Every production chatbot is a version of this class — with more error handling, retry logic, and streaming.

*This cell defines the class but produces no output. Run it, then continue to C2.*


In [ ]:
# Cell C1 — ChatSession: the stateful conversation pattern
class ChatSession:
    '''Tracks conversation history. Stateful chat = history accumulation.'''
    def __init__(self, system_prompt='You are a helpful assistant.'):
        self.history = [{'role': 'system', 'content': system_prompt}]

    def chat(self, user_message, max_new_tokens=90, warn_at=0.8):
        self.history.append({'role': 'user', 'content': user_message})
        used = self.token_estimate(add_generation_prompt=True)
        limit = config.max_position_embeddings
        if used / limit >= warn_at:
            print(f'WARNING: context is at {used/limit:.1%} of the model window ({used}/{limit} tokens).')

        formatted = tokenizer.apply_chat_template(
            self.history, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(formatted, return_tensors='pt')
        input_len = inputs['input_ids'].shape[1]
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        answer = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
        self.history.append({'role': 'assistant', 'content': answer})
        return answer

    def token_estimate(self, add_generation_prompt=False):
        '''Exact tokenizer count for the full formatted chat prompt.'''
        formatted = tokenizer.apply_chat_template(
            self.history, tokenize=False, add_generation_prompt=add_generation_prompt
        )
        return len(tokenizer(formatted)['input_ids'])

    def show(self):
        icons = {'system': '[system]', 'user': '[user]', 'assistant': '[assistant]'}
        for m in self.history:
            preview = m['content'][:100] + ('...' if len(m['content']) > 100 else '')
            print(f"{icons[m['role']]} {m['role'].upper()}: {preview}")

    def show_full_prompt(self):
        '''Print the exact prompt string sent to the local model.'''
        print(tokenizer.apply_chat_template(
            self.history, tokenize=False, add_generation_prompt=True
        ))

    def trim(self, max_turns=3):
        '''Keep system prompt plus the last N user/assistant pairs.'''
        system = self.history[:1]
        turns = self.history[1:]
        keep_messages = max_turns * 2
        self.history = system + turns[-keep_messages:]


### Step C2 — A Three-Turn Conversation

This cell runs three questions in sequence. After each answer, it prints `[Context: ~N tokens]`.

**Watch the token count grow with each turn.** That growing number is the full history being re-sent every time.

After the conversation, `bot.show_full_prompt()` prints the complete raw string that was sent on the final turn — you will see all three prior turns formatted with `<|im_start|>` tokens, followed by the new user message.

*This is the "aha" moment for most students: the model received a 300-token prompt just to answer one short question.*


In [ ]:
# Cell C2 — Run a three-turn conversation
bot = ChatSession(
    system_prompt='You are an LLM deployment expert. Max 2 sentences per answer.'
)

print(bot.chat('What is quantization?'))
print(f'[Context: ~{bot.token_estimate()} tokens]\n')
print('─' * 40)
print(bot.chat('How does it compare to pruning?'))
print(f'[Context: ~{bot.token_estimate()} tokens]\n')
print('─' * 40)
print(bot.chat('Which should I try first when deploying a 7B model?'))
print(f'[Context: ~{bot.token_estimate()} tokens]\n')
print()
bot.show()

print('\nFULL PROMPT SENT TO THE MODEL:')
bot.show_full_prompt()


### Step C3 — Context Window as a Production Constraint

The output of C2 showed ~289 tokens used out of a 32,768-token limit — only 0.88%. That looks fine.

But imagine a real support bot:
- 50 turns × ~200 tokens per turn = 10,000 tokens
- Add a long system prompt and document context → you hit the limit fast
- When you hit it: **the model silently truncates the oldest messages** — it does not warn you, it does not error — it just forgets

This cell prints your current status and lists the four production strategies for handling a full context window. Read each one and think about the trade-off it makes.


In [ ]:
# Cell C3 — Context window as a production constraint
context_limit = config.max_position_embeddings
used          = bot.token_estimate()

print('CONTEXT WINDOW STATUS')
print(f'  Used      : ~{used:,} tokens')
print(f'  Limit     :  {context_limit:,} tokens  ({MODEL_ID})')
print(f'  Remaining : ~{context_limit - used:,} tokens')
print(f'  Pressure  :  {used/context_limit*100:.2f}%')
print()
print('When context fills in production:')
print('  Option A — Sliding window: drop oldest turns')
print('  Option B — Summarize:      compress old turns into a system prompt update')
print('  Option C — Retrieve:       keep old facts in a vector store and retrieve relevant pieces')
print('  Option D — New session:    carry a summary forward to a fresh history')
print()
print('Each option trades cost, latency, and continuity. There is no free lunch.')


#### ✅ Part C Checkpoint

- [ ] You can explain why token count grows each turn
- [ ] You have seen the full raw prompt printed — it contains all history
- [ ] You understand the four options when a context window fills


---

## ✏️ Required Exercise — Sliding Window Trim

You have seen the context grow. Now fix it.

**The task:** The `trim(max_turns=N)` method keeps the system prompt plus the last N user/assistant pairs and discards everything older. Call it and verify the token count drops.

**Why this matters:** In production, you cannot let history grow unbounded. A sliding window is the simplest strategy — but notice what you lose: the model will no longer "remember" anything from the turns that were dropped. This is the core trade-off.

*Run the cell below and confirm:*
- Before trim: N messages\n
- After trim to 1 turn: significantly fewer tokens\n
- The middle two turns are gone from the history


In [ ]:
print('Before trim:')
print(f'  messages: {len(bot.history)}')
print(f'  tokens  : {bot.token_estimate()}')

bot.trim(max_turns=1)

print('\nAfter trim to last 1 turn:')
print(f'  messages: {len(bot.history)}')
print(f'  tokens  : {bot.token_estimate()}')
print()
bot.show()


#### 💬 Reflection Question

After trimming to 1 turn, if you ask the bot "What were we just talking about?" — what will it answer? Why?

*(Think about this before checking. The answer is: it will not know, because those turns were deleted from its history. It has no other memory.)*


---

## ✅ Lab 3 Complete

You should now be able to answer all of these:
- [ ] Architecture numbers for Qwen2.5-0.5B (hidden size, layers, context window)
- [ ] Greedy decoding: two identical outputs
- [ ] Temperature 0.1 is conservative; 1.5 is creative and sometimes incoherent
- [ ] Chat template printed — raw string, user-only, and system+user formats
- [ ] Multi-turn conversation with growing token count
- [ ] Context window status printed
- [ ] Sliding-window trim exercise completed — and you understand what is lost when turns are dropped

---

## 🧠 Key Takeaways

1. **A model is a file** — a fixed shape with a config and weights. You can inspect both.
2. **Generation is sampling** — temperature and top-p are product controls, not magic.
3. **Chat templates are a contract** — skipping them breaks the model's trained behavior.
4. **Chat memory is manual** — you build and manage the history list yourself.
5. **Context windows fill** — and when they do, the model silently forgets.

## Stretch Goals

1. **Config comparison:** Load only the config (no weights) for `microsoft/phi-4` and `google/gemma-2-2b-it`.
   Compare their context windows and hidden sizes. Which would you choose for a 2K-token RAG prompt?
2. **Structured output:** Add `response_format={'type': 'json_object'}` to a chat call. 
   Ask the model to return its answer as JSON with keys `answer` and `confidence`. Parse and print.
3. **Top-k:** Modify the generate helper to accept `top_k`. Compare outputs at `top_k=20` vs `top_k=100` on the same deployment prompt.
